# Časť 1, Téma 2: Preskočenie hesla glitchingu s hodinami

---
POZNÁMKA: Toto cvičenie odkazuje na niektoré (komerčné) školiace materiály na stránke ChipWhisperer.io. Cvičenie môžete voľne spúšťať a používať v súlade s licenciou open source (vrátane použitia vo vašich vlastných kurzoch, ak ich distribuujete podobným spôsobom), musíte však zachovať odkaz na tento zdroj. Zvážte účasť na našom školiacom kurze, aby ste si mohli vychutnať plnohodnotný zážitok.

---

**Zhrnutie:** *V predchádzajúcom cvičení sme sa naučili, ako možno pomocou glitchingu hodín spôsobiť neočakávané správanie cieľového zariadenia. V tomto cvičení sa pozrieme na o niečo realistickejší príklad – glitching s cieľom obísť kontrolu hesla*

**Čo sa naučíme:**

* Uplatnenie predchádzajúcich nastavení glitchingu na nový firmvér
* Kontrola úspešnosti a neúspešnosti pri glitchingu

## Firmware

Už sme videli, ako môžeme vkladať hodinové glitche, aby sme narušili výpočet, ktorý sa cieľ snaží vykonať. Hoci to má mnoho uplatnení, z ktorých niektoré rozoberieme neskôr, pozrime sa na niečo, čo je o niečo bližšie k nášmu pôvodnému príkladu kódu citlivého na glitche: kontrolu hesla. Tu nie je potrebné meniť firmware, stále používame projekt simpleserial-glitch (hoci si znovu prejdeme všetky kroky zostavenia).

Kód pre kontrolu hesla je nasledovný:

```C
uint8_t password(uint8_t* pw)
{
    char passwd[] = "touch";
    char passok = 1;
    int cnt;

    trigger_high();

    //Simple test - nekonrolujeme príliš dlhé heslo
    for(cnt = 0; cnt < 5; cnt++){
        if (pw[cnt] != passwd[cnt]){
            passok = 0;
        }
    }
    
    trigger_low();
    
    simpleserial_put('r', 1, (uint8_t*)&passok);
    return passok;
}
```

Nie je tu vlastne nič mimoriadne – ide len o jednoduchú kontrolu hesla. Môžeme s ním komunikovať pomocou príkazu `‚p‘`.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
SS_VER = 'SS_VER_2_1'

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
fw_path = "../../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER == 'SS_VER_2_1':
    target.reset_comms()

In [ ]:
def reboot_flush():
    reset_target(scope)
    #Flush garbage too
    target.flush()

Ak pošleme zlé heslo

In [ ]:
#Do glitch loop
reboot_flush()pw = bytearray([0x00]*5)
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

Dostaneme v odpovedi nulu. Ale ak pošleme správne heslo:

In [ ]:
#Do glitch loop
reboot_flush()
pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68]) # correct password ASCII representation
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

Dostaneme naspäť 1. Teraz nastavíme glitch ako naposledy:

In [ ]:
scope.cglitch_setup()

Uprav tento kód a pridáme ext offset parameter:

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

Rovnako ako predtým, znázorníme naše nastavenia. Graf je dvojrozmerný, takže sa budete musieť rozhodnúť, čo chcete znázorniť. V tomto prípade znázorníme offset a ext_offset, ale vyberte si, čo sa vám páči:

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="ext_offset", y_index="offset")

A vytvorte glitchovú slučku. Nezabudnite použiť niektoré osvedčené nastavenia, ktoré ste našli v predchádzajúcom cvičení, pretože vďaka tomu bude toto cvičenie oveľa kratšie!

Jednou zo zmien, ktorú budete pravdepodobne chcieť urobiť, je pridať vyhľadávanie ext_offset. Počet miest, kde môžeme vložiť úspešný glitch, sa výrazne znížil. Tento krok bude veľmi dôležitý aj pre budúce cvičenia.

In [ ]:
from tqdm.notebook import tqdm
import re
import struct
gc.set_global_step(step)

# replace these with your settings from the last part
# The example width/offset settings shown here are for CW-Lite/Pro; width/offset are expressed differently for Husky (see Fault 1_1)
gc.set_range("width", ???, ???)
gc.set_range("offset", ???, ???)
gc.set_global_step(???)

# you can leave this range for ext_offset
gc.set_range(???, 0, 100)
gc.set_step(???, 1)

# adjust if you want
step = 1

scope.glitch.repeat = 1
reboot_flush()

for glitch_settings in gc.glitch_values():
    scope.glitch.offset = ???
    scope.glitch.width = ???
    scope.glitch.ext_offset = ???
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        print("Trigger still high!")
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()

    scope.arm()
    target.simpleserial_write('p', bytearray(???)) #send an incorrect password

    ret = scope.capture()

    val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50) #For loop check
    if ret:
        print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()

    else:
        if invalid_response: #fill in with invalid response detection
            gc.add("reset")
        else:
            if glitched_past_pw_check: # replace this
                gc.add("success")
                print(val['payload'])
                print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                print("🐙", end="")
            else:
                gc.add("normal")

Pre toto cvičenie budete pravdepodobne potrebovať dve kópie svojich výsledkov; jednu na určenie efektívnych ext_offsets:

In [ ]:
results = gc.calc(ignore_params=["width", "offset"], sort="success_rate")
results

A ešte jedna pre nastavenie "width"/"offset":

In [ ]:
results = gc.calc(ignore_params=["ext_offset"], sort="success_rate")
results

In [ ]:
scope.dis()
target.dis()